# DataBricks と Numpy の相性は最悪に近い
- DataBricks と Numpy の相性は最悪に近い(Repeat!!!)
- Kernel Crush が発生しやすく、回避が困難である
- そのため、Deltalake TableおよびSparkを用いた構成に書き換えることとした

In [ ]:
%pip install                                                 \
        numpy                 adlfs               httpx      \
        python-dotenv         openai              polars     \
		azure-ai-inference    packaging           jageocoder

%restart_python

In [ ]:
import os
import gc
import ast
import asyncio
import json
import httpx
import jageocoder
import numpy  as np
import scipy  as sp
from typing       import Literal, cast
from pathlib      import Path
from tqdm         import tqdm
from tqdm.asyncio import tqdm as asynctqdm
from dotenv       import load_dotenv
from openai       import AsyncOpenAI

import pandas as pd
from pyspark.sql import SparkSession, Window, types
from pyspark.sql.functions import col
import pyspark.sql.functions as F
from delta       import configure_spark_with_delta_pip

from azure.ai.inference.models import SystemMessage, UserMessage

from llm_agent   import LlmAgent


In [ ]:
builder = SparkSession.builder\
            .config("spark.sql.sources.commitProtocolClass", "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")\
            .config("spark.sql.parquet.output.committer.class", "org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter")\
            .config("spark.mapreduce.fileoutputcommitter.marksuccessfuljobs","false")\
            .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")\
            .config("spark.sql.adaptive.enabled", True)\
            .config("spark.sql.shuffle.partitions", "auto")\
            .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "100MB")\
            .config("spark.sql.adaptive.coalescePartitions.enabled", True)\
            .config("spark.sql.dynamicPartitionPruning.enabled", True)\
            .config("spark.sql.autoBroadcastJoinThreshold", "10MB")\
            .config("spark.sql.session.timeZone", "Asia/Tokyo")\
            .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")\
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
            .config("spark.databricks.delta.write.isolationLevel", "SnapshotIsolation")\
            .config("spark.databricks.delta.optimizeWrite.enabled", True)\
            .config("spark.databricks.delta.autoCompact.enabled", True)
            # Delta Lake 用の SQL コミットプロトコルを指定
            # Parquet 出力時のコミッタークラスを指定
            # Azure Blob Storage (ABFS) 用のコミッターファクトリを指定
            # '_SUCCESS'で始まるファイルを書き込まないように設定
            # JavaSerializerからKryoSerializerへの変更
            # AQE(Adaptive Query Execution)の有効化
            # パーティション数を自動で調整するように設定
            # シャッフル後の1パーティションあたりの最小サイズを指定
            # AQEのパーティション合成の有効化
            # 動的パーティションプルーニングの有効化
            # 小さいテーブルのブロードキャスト結合への自動変換をするための閾値調整
            # SparkSessionのタイムゾーンを日本標準時刻に設定
            # Apache Arrow 形式で Pandas<=>Spark 変換を行う
            # Delta Lake固有のSQL構文や解析機能を拡張モジュールとして有効化
            # SparkカタログをDeltaLakeカタログへ変更
            # Delta Lake書き込み時のアイソレーションレベルを「スナップショット分離」に設定
            # 書き込み時にデータシャッフルを行い、大きなファイルを生成する機能の有効化
            # 書き込み後に小さなファイルを自動で統合する機能の有効化

# 追加の設定処理
# クラスターサイズによって、動的に書き換えること
builder = builder\
			.config("spark.driver.memory", "64g")\
    		.config("spark.driver.maxResultSize", "32g")\
			.config("spark.rpc.message.maxSize", "1024")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# メモリ管理
del builder
gc.collect()


In [ ]:
# .env ファイルを読み込む
load_dotenv()

# 環境変数の取得
AI_FOUNDRY_ENDPOINT       = os.environ.get("AI_FOUNDRY_ENDPOINT")
AI_FOUNDRY_API_KEY        = os.environ.get("AI_FOUNDRY_API_KEY")
AI_FOUNDRY_MODEL          = os.environ.get("AI_FOUNDRY_MODEL")
LLM_MAX_TOKENS            = os.environ.get("LLM_MAX_TOKENS")
LLM_TEMPERATURE           = os.environ.get("LLM_TEMPERATURE")
LLM_TOP_P                 = os.environ.get("LLM_TOP_P")

# ウェジットからの設定の読み込み
BRAND_NAME                = dbutils.widgets.get('BRAND_NAME')
TARGET_BRAND_WORDS        = dbutils.widgets.get('TARGET_BRAND_WORDS')
EXTRACTION_ID             = dbutils.widgets.get('EXTRACTION_ID')

# メモ：
# 環境変数・ウィジット経由でのパラメータの取得方法では、無条件にパラメータの型がStringに変換されてしまう
# そのため、受け取ったパラメータを適切に型変換する必要がある
LLM_MAX_TOKENS       = int(LLM_MAX_TOKENS)
LLM_TEMPERATURE      = float(LLM_TEMPERATURE)
LLM_TOP_P            = float(LLM_TOP_P)
EXTRACTION_ID        = int(EXTRACTION_ID)

# メモ：
# TARGET_BRAND_WORDS の中身として
# "['aaa', 'bbb', 'ccc']"
# "aaa" or "'aaaaa'"
# 上記のような「文字列」を想定している
# 
# このような文字列を正しくパースするために、astライブラリを利用することとした
try:
	TARGET_BRAND_WORDS = ast.literal_eval(TARGET_BRAND_WORDS)
	if isinstance(TARGET_BRAND_WORDS, str):
		TARGET_BRAND_WORDS = [TARGET_BRAND_WORDS]
except (ValueError, SyntaxError):
	TARGET_BRAND_WORDS = [TARGET_BRAND_WORDS]

# 簡易デバッグ用
# print(f'AI_FOUNDRY_ENDPOINT:       {AI_FOUNDRY_ENDPOINT}')       # セキュリティリスクのため、コメントアウト
# print(f'AI_FOUNDRY_API_KEY:        {AI_FOUNDRY_API_KEY}')        # セキュリティリスクのため、コメントアウト
print(f'AI_FOUNDRY_MODEL:          {AI_FOUNDRY_MODEL}')
print(f'LLM_MAX_TOKENS:            {LLM_MAX_TOKENS}')
print(f'LLM_TEMPERATURE:           {LLM_TEMPERATURE}')
print(f'LLM_TOP_P:                 {LLM_TOP_P}')
print(f'EXTRACTION_ID:             {EXTRACTION_ID}')

In [ ]:
limits      = httpx.Limits(max_keepalive_connections=20, max_connections=300)
timeout     = httpx.Timeout(300.0, connect=5.0)
http_client = httpx.AsyncClient(limits=limits, timeout=timeout)
openai_cli  = AsyncOpenAI(
                    base_url=AI_FOUNDRY_ENDPOINT,
                    api_key=AI_FOUNDRY_API_KEY,
                    http_client=http_client,
                    max_retries=3
                )
semaphore   = asyncio.Semaphore(50)
llmClient   = LlmAgent(openai_cli, AI_FOUNDRY_MODEL, int(LLM_MAX_TOKENS), float(LLM_TEMPERATURE), float(LLM_TOP_P), semaphore)



In [ ]:
output_file_path = f'ブランドLP/{BRAND_NAME}.md'

# 商品分析情報が存在しない場合
if not os.path.exists(output_file_path):
	with open(f'ブランドLP/商品分析.md', 'r', encoding='utf-8') as f:
		sys_prompt = f.read()
		sys_prompt = sys_prompt.replace('【WORD_LIST】', ', '.join(TARGET_BRAND_WORDS))

	messages  = []
	messages.append(SystemMessage(content=sys_prompt))
	product_analysis = await llmClient.complete(messages)
	with open(output_file_path, 'w', encoding='utf-8') as f:
		f.write(json.dumps(product_analysis, indent=4, ensure_ascii=False))

# 商品分析情報の読み込み
with open(output_file_path, 'r', encoding='utf-8') as f:
    product_analysis = f.read()
product_analysis

In [ ]:
# メモ：
# cohort.npzは以下のようなデータ構成になっている
# cohort.npz
#     |-- data              : 計算済みコホート係数行列
#     |-- indices           : データの位置指定子(列)
#     |-- indptr            : 行ごとのスライスインデックス
#     |-- shape             : コホート係数行列の形(ADIDのリスト × コホートキャプションのリスト)
#     |-- adid_list         : ADIDのリスト(列)
#     |-- business_codelist : コホートキャプションIDのリスト(行)
TARGET      = "/Volumes/stgadintedmpadintedi/featurestore/behaviorvector/cohort.npz"
npz         = np.load(TARGET, allow_pickle=True)
np_cohort   = sp.sparse.csr_matrix((npz["data"], npz["indices"], npz["indptr"]), shape=tuple(npz["shape"]))
np_adidlist = npz["adid_list"]

# メモリ管理
del npz
gc.collect()

print(np_cohort.shape)

In [ ]:
# 特定の地点に対するADIDリストを取得
sdf_numpylist     = spark.createDataFrame(pd.DataFrame({'adid': np_adidlist}))
POLYGONLIST_TABLE = "adinte_adintedmp.envprod.polygonlist"
sdf_polygonlist   = spark.read.table(POLYGONLIST_TABLE)\
							.select(['project', 'caption', 'extraction_id', 'branch_id', 'spot_id', 'lat_top', 'lon_left'])\
                            .filter(col('extraction_id') == EXTRACTION_ID)\
                            .dropDuplicates()

# メモ：
# sourceテーブルはレコードに一定期間の寿命が付与されている
# 予め決められた期間を過ぎるとレコードが削除される仕組みであるようだ
# そのため、できる限りローカルにバックアップデータを保存しておいて、これを参照する形式にすることとした
ADIDLIST_PATH = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/preprocess/source_table_{EXTRACTION_ID}.parquet"
path_sdf_file = Path(ADIDLIST_PATH).resolve()
if not path_sdf_file.exists():
	ADIDLIST_TABLE = "adinte_adintedmp.envprod.source"
	pdf_adidlist   = spark.read.table(ADIDLIST_TABLE)\
								.filter(col('extraction_id') == EXTRACTION_ID)\
								.toPandas()
	pdf_adidlist.to_parquet(ADIDLIST_PATH, compression="snappy")
	del pdf_adidlist
	gc.collect()

sdf_adidlist = spark.read.parquet(str(path_sdf_file))\
							.select(['extraction_id', 'branch_id', 'spot_id', 'adid'])\
							.filter(col('extraction_id') == EXTRACTION_ID)\
							.join(sdf_numpylist,   on=['adid'], how='inner')\
							.join(sdf_polygonlist, on=['extraction_id', 'branch_id', 'spot_id'], how='inner')\
							.select(['caption', 'adid', 'lat_top', 'lon_left'])\
							.dropDuplicates()

# メモリ管理
sdf_numpylist.unpersist()
sdf_polygonlist.unpersist()
del sdf_numpylist, sdf_polygonlist
spark.catalog.clearCache()
gc.collect()

sdf_adidlist.display()

In [ ]:
# メモリ節約のための変数管理
pdf_adidlist    = sdf_adidlist.select('adid').dropDuplicates().toPandas()
target_adidlist = pdf_adidlist['adid'].tolist()


# 特定のADIDのみの抜き出し
indexer        = pd.Index(np_adidlist)
indices        = indexer.get_indexer(target_adidlist)
# indices        = indices[indices != -1]
np_cohort      = np_cohort[indices, :]

del target_adidlist
del indexer, indices
gc.collect()

# メモ：
# L2正規化を行うにあたって、疎行列専用に書く必要がある
# 密行列であれば簡単に書くことが可能であるが、今回の要件に適合しない
# そのため疎行列な対角行列を経由することにより、これを実現することとした
# 
# 当初すでにL2正規化は適用済みであったが、時々適用されていない状態になることがあるようだ
# そのため、改めて適用することとした
# L2正規化
if not np.allclose(np_cohort.power(2).sum(axis=1), 1.0):
	l2norm_vector = sp.sparse.linalg.norm(np_cohort, axis=1)
	np_cohort     = sp.sparse.diags(1 / np.maximum(l2norm_vector, 1e-10)) @ np_cohort

	# メモリ管理
	del l2norm_vector
	gc.collect()



In [ ]:
# CODELISTに対応するキャプションの文脈行列を取得
CAPTION_CONTEXT_MATRIX = "cohort_caption_matrix.npz"
npz                    = np.load(CAPTION_CONTEXT_MATRIX, allow_pickle=True)
relational_spots       = npz["business_placelist"]

# Sparkフレームワークに変換する
np_cohort.data = np.exp(np_cohort.data)
inv_sums       = 1.0 / np.maximum(np_cohort.sum(axis=1).A1, 1e-10)
np_cohort      = sp.sparse.diags(inv_sums) @ np_cohort
coo_cohort     = np_cohort.tocoo()

# メモリ管理
del npz
del inv_sums
del np_cohort
gc.collect()

joined_spots   = np.array(['_'.join(spot) for spot in relational_spots])

# メモ：
# 当初は、比較的に簡単に簡潔に処理を書いていた
# しかし、この処理ではDatabricks Jobがクラッシュすることが判明した
# より具体的には、Jobの裏側で動作しているJVMである
# クラッシュの原因は一つのマスターノードに巨大なSparkフレームワークが展開されたことにある
# ワーカーノードに転送される前にメモリが枯渇し、事実上のクラッシュ状態に陥る
# pdf_cohort     = pd.DataFrame({
# 						'adid'           : np_adidlist[ coo_cohort.row], 
# 						'cohort_caption' : joined_spots[coo_cohort.col], 
# 						'probability'    : coo_cohort.data
# 					})
# sdf_cohort     = spark.createDataFrame(pdf_cohort)
# 
# これの回避のために、Sparkフレームワークを3つに分割することとした
# 分割されたフレームワークを1つに再統合することで、メモリ枯渇を回避することが目的である
sdf_cohort_idx = spark.createDataFrame(pd.DataFrame({
						'row_idx'     : coo_cohort.row, 
						'col_idx'     : coo_cohort.col, 
						'probability' : coo_cohort.data
					}))
sdf_adid_idx   = spark.createDataFrame(pd.DataFrame({
						'row_idx'        : range(len(np_adidlist)),
						'cohort_caption' : np_adidlist,
					}))
sdf_spot_idx   = spark.createDataFrame(pd.DataFrame({
						'col_idx'        : range(len(joined_spots)),
						'cohort_caption' : joined_spots,
					}))
sdf_cohort     = sdf_cohort_idx\
					.join(sdf_adid_idx, on='row_idx', how='inner')\
                    .join(sdf_spot_idx, on='col_idx', how='inner')\
                    .select(['adid', 'cohort_caption', 'probability'])

# メモリ管理
sdf_cohort_idx.unpersist()
sdf_adid_idx.unpersist()
sdf_spot_idx.unpersist()
del np_adidlist, relational_spots
del coo_cohort, joined_spots
del sdf_cohort_idx, sdf_adid_idx, sdf_spot_idx
spark.catalog.clearCache()
gc.collect()

sdf_cohort.display()

In [ ]:
# 突貫のため、精度無視
# 突貫のため、精度無視
# 突貫のため、精度無視
# 突貫のため、精度無視
# 突貫のため、精度無視


# 上位50位までのAIICOスコアを取得
AIICO_SCORE_PATH = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME}_caption_score.parquet"
sdf_aiico_score  = spark.createDataFrame(pd.read_parquet(AIICO_SCORE_PATH))\
						.select(['caption', 'score', 'lat_top', 'lon_left'])\
						.orderBy('score', ascending=False)\
                        .limit(50)

# AIICO毎の性別・年齢層を取得
window_spec      = Window.partitionBy('caption')
AGE_GENDER_TABLE = "adinte_datainfrastructure.master.mobilewalla_agegender"
sdf_agegender    = spark.read.table(AGE_GENDER_TABLE)\
						.select( ['adid', 'age', 'gender'])\
                        .join(sdf_adidlist, on='adid', how='inner')\
                        .select( ['caption', 'adid', 'age', 'gender'])\
                        .groupBy(['caption', 'age', 'gender'])\
                        .agg(F.count('*').alias('count'))
# メモ：
# np_cohort 行列はL2正規化を施した疎行列である
# そのため、softmax関数を適用することにより確率として解釈しようとしている
# 
# AIICO毎の行動ベクトルを取得
sdf_behavioral   = sdf_cohort\
                        .select(['adid', 'cohort_caption', 'probability'])\
                        .join(sdf_adidlist, on='adid', how='inner')\
                        .select( ['caption', 'adid', 'cohort_caption', 'probability'])\
                        .groupBy(['caption', 'cohort_caption'])\
                        .agg(F.sum('probability').alias('sum_probability'))\
                        .withColumn('probability', col('sum_probability') / F.sum('sum_probability').over(window_spec))\
                        .select( ['caption', 'cohort_caption', 'probability'])


# 全体分析に必要なフレームワーク生成
sdf_agegender_overall  = sdf_agegender\
							.select( ['age', 'gender', 'count'])\
							.groupBy(['age', 'gender'])\
                            .agg(F.count('count').alias('count'))\
                            .withColumn('probability', col('count') / F.sum('count'))\
                        	.select( ['age', 'gender', 'probability'])
sdf_behavioral_overall = sdf_behavioral\
							.select( ['cohort_caption', 'probability'])\
                            .groupBy(['cohort_caption'])\
                            .agg(F.sum('probability').alias('sum_probability'))\
                            .withColumn('probability', col('sum_probability') / F.sum('sum_probability'))\
                            .select( ['cohort_caption', 'probability'])

# 詳細分析に必要なフレームワーク生成
sdf_agegender_specific  = sdf_agegender\
							.select( ['caption', 'age', 'gender', 'count'])\
							.join(sdf_aiico_score.limit(10), on='caption', how='inner')\
							.groupBy(['caption', 'age', 'gender'])\
                            .agg(F.count('count').alias('count'))\
                            .withColumn('probability', col('count') / F.sum('count').over(window_spec))\
                        	.select( ['caption', 'age', 'gender', 'probability'])
sdf_behavioral_specific = sdf_behavioral\
							.select( ['caption', 'cohort_caption', 'probability'])\
                            .join(sdf_aiico_score.limit(10), on='caption', how='inner')\
                            .groupBy(['caption', 'cohort_caption'])\
                            .agg(F.sum('probability').alias('sum_probability'))\
                            .withColumn('probability', col('sum_probability') / F.sum('sum_probability').over(window_spec))\
                            .select( ['caption', 'cohort_caption', 'probability'])

# メモリ管理
sdf_adidlist.unpersist()
sdf_cohort.unpersist()
sdf_agegender.unpersist()
sdf_behavioral.unpersist()
del sdf_adidlist, sdf_cohort
del sdf_agegender, sdf_behavioral
spark.catalog.clearCache()
gc.collect()

sdf_behavioral_overall.display()

In [ ]:
# LLM用プロンプトの作成のための前処理
@F.udf(returnType=types.StringType())
def get_address(latitude:float, longitude:float) -> str:
	jageocoder.init(url='https://jageocoder.info-proto.com/jsonrpc')
	location = jageocoder.reverse(longitude, latitude, level=8, as_dict=True)
	if len(location) == 0:
		return ''
	
	location_dict = cast(dict, location[0])
	address       = location_dict['candidate']['fullname']
	return address[0]

sdf_aiico_score_top_50       = sdf_aiico_score\
									.withColumn('prefecture', get_address(col('lat_top'), col('lon_left')))\
									.selelct(['prefecture', 'caption', 'score'])
sdf_agegender_specific_prep  = sdf_agegender_specific\
									.withColumn('str_agegender', F.format_string("'%s' × '%s'", F.col("age"), F.col("gender")))\
									.withColumn('str_agegender', F.format_string('(%s, %s)', F.col('str_agegender'), F.col('probability')))\
									.select(['caption', 'str_agegender'])\
									.groupBy('caption')\
									.agg(F.collect_list('str_agegender').alias('list_agegender'))\
									.select(['caption', 'list_agegender'])
sdf_behavioral_specific_prep = sdf_behavioral_specific\
									.withColumn('str_behavioral', F.format_string('(%s, %s)', F.col('cohort_caption'), F.col('probability')))\
									.select(['caption', 'str_behavioral'])\
									.groupBy('caption')\
									.agg(F.collect_list('str_behavioral').alias('list_behavioral'))\
									.select(['caption', 'list_behavioral'])
sdf_llm_prompt_prep          = sdf_agegender_specific_prep\
									.join(sdf_behavioral_specific_prep, on='caption', how='inner')\
									.select(['caption', 'list_agegender', 'list_behavioral'])

# メモリ管理
sdf_aiico_score.unpersist()
sdf_agegender_specific.unpersist()
sdf_behavioral_specific.unpersist()
sdf_agegender_specific_prep.unpersist()
sdf_behavioral_specific_prep.unpersist()
del sdf_aiico_score, sdf_agegender_specific, sdf_behavioral_specific
del sdf_agegender_specific_prep, sdf_behavioral_specific_prep
spark.catalog.clearCache()
gc.collect()

content = (
import json

# --- 事前準備：Spark DataFrameをLLMが読みやすいJSON文字列に変換 ---
# ※ toPandas() はデータ量がLLMのコンテキストウィンドウに収まる前提です
data_top_50        = sdf_aiico_score_top_50.toPandas().to_json(orient='records', force_ascii=False)
data_age_gen_all   = sdf_agegender_overall.toPandas().to_json(orient='records', force_ascii=False)
data_behav_all     = sdf_behavioral_overall.toPandas().to_json(orient='records', force_ascii=False)
data_detailed      = sdf_llm_prompt_prep.toPandas().to_json(orient='records', force_ascii=False)

# --- プロンプト定義 ---
content = f"""
# 前提条件
あなたは外資系コンサルティングファーム出身の、極めて優秀なプロダクトマーケティングマネージャー兼リテール戦略アナリストです。
以下の【提供データ】を分析し、指定された【出力形式】と【レポート構成】に従って、データドリブンな戦略レポートを作成してください。

# 出力形式の厳格なルール
- 出力は完全なHTML形式（<!DOCTYPE html>から始める）のみとしてください。
- Markdownのコードブロック（```html など）や、HTML以外の解説テキストは一切出力しないでください。
- レポートとして視覚的に分かりやすくなるよう、<style>タグを用いてプロフェッショナルで洗練されたCSSデザイン（テーブルの罫線、見出しの装飾、余白など）を適用してください。

# 提供データ
## 1. コア商材情報
- ブランド・商材名: {BRAND_NAME}

## 2. ランキングデータ（Top 50地点）
{data_top_50}

## 3. 全体分析データ（マクロ傾向）
- 属性全体（年齢・性別）: {data_age_gen_all}
- 行動全体（回遊先）: {data_behav_all}

## 4. 詳細分析データ（地点ごとのミクロ傾向）
{data_detailed}

# レポート構成
以下の3つのセクションを持つHTMLレポートを構築してください。各項目は抽象論を避け、データに基づく具体的な仮説と提案に落とし込んでください。

## セクション1：ターゲット地点 Top 50 ランキング
- 「ランキングデータ」を使用し、スコア順に整理された見やすいHTMLテーブル（順位、地点名、住所、スコアなど）を作成してください。
- テーブルの直下に、上位地点のロケーション特性に関する共通点や傾向を1〜2段落の短いエグゼクティブサマリーとして添えてください。

## セクション2：全体分析（マクロ戦略）
- 「全体分析データ」を使用し、ターゲットエリア全体に訪れる顧客の「メインペルソナ（ライフスタイル、価値観）」と、1日の行動導線である「カスタマージャーニー（どのような目的で回遊しているか）」の仮説を定義してください。
- 商材（{BRAND_NAME}）と、このマクロな顧客特性がどのようにマッチしているか（プロダクト・ロケーション・フィット）、独自の提供価値を言語化してください。

## セクション3：詳細分析とアクションプラン（ミクロ戦略）
- 「詳細分析データ」を使用し、地点ごとの特性の違い（例：若年層中心の地点 vs ファミリー層中心の地点、回遊先が商業施設 vs 観光地 など）を分析し、いくつかの「店舗タイプ（セグメント）」に類型化・グルーピングしてください。
- 類型化したセグメントごとに、以下の具体的なマーケティングアクションを提案してください。
  1. **店舗レイアウト・MD提案:** 判明したペルソナと回遊目的を踏まえた、商品展開や陳列、POPのメッセージング。
  2. **コラボ＆送客戦略:** よく回遊されている周辺施設（競合や親和性の高いスポット）との連携やクロスセル施策。
  3. **広告・販促アプローチ:** この客層に対して、いつ・どのタイミングでプロモーションを打つべきかの具体案。
"""
		)

# メモリ管理
sdf_llm_prompt_prep.unpersist()
del sdf_llm_prompt_prep
spark.catalog.clearCache()
gc.collect()

messages = []
messages.append(SystemMessage(content=content))
messages.append(UserMessage(content=product_analysis))
res = await llmClient.complete(messages)

res